# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. output: {'A': 2, 'B': 1} буква 'A' посчитается дважды, а 'B' — один раз.
2. output: [0, 1, 1] f(a) каждый раз меняет сам исходный список a на месте, добавляя в него 1.
3. output: 2 set автоматом отбрасывает дубликаты и оставляет только 2 уникальных элемента.
4. output: {'ab': 2, 'abc': 3} старое значение просто перезаписывается
5. output: 5; None  


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# База
#1
def to_num(s):
    if not s:
        return 0.0
    cleaned = str(s).replace(" ", "").replace("₸", "").strip()
    try:
        return float(cleaned)
    except ValueError:
        return 0.0


CITY_MAP = {
    "almaty": "Алматы",
    "алматы": "Алматы",
    "astana": "Астана",
    "астана": "Астана",
    "shymkent": "Шымкент",
    "шымкент": "Шымкент",
    "karaganda": "Караганда",
    "караганда": "Караганда",
    "aktobe": "Актобе",
    "актобе": "Актобе",
    "atyrau": "Атырау",
    "атырау": "Атырау",
}


def clean_city(s):
    key = str(s).strip().lower()
    return CITY_MAP.get(key, str(s).strip().title())

#2
cat_counts = {}
for row in rows:
    cat = row["category"].strip()
    cat_counts[cat] = cat_counts.get(cat, 0) + 1

sorted_cats = sorted(cat_counts.items(), key=lambda x: x[1], reverse=True)

for cat, cnt in sorted_cats:
    print(f"{cat}: {cnt}")

#3
cat_revenue = {}
cat_paid_count = {}

for row in rows:
    status = row["status"].strip().lower()
    if status in ("paid", "оплачен"):
        cat = row["category"].strip()
        amount = to_num(row["amount"])
        
        cat_revenue[cat] = cat_revenue.get(cat, 0.0) + amount
        cat_paid_count[cat] = cat_paid_count.get(cat, 0) + 1

print(f"{'Категория':<20} | {'Оплачено':>10} | {'Выручка, ₸':>15} | {'Ср. чек, ₸':>12}")
print("-" * 65)

for cat in sorted(cat_revenue, key=cat_revenue.get, reverse=True):
    rev = cat_revenue[cat]
    cnt = cat_paid_count[cat]
    avg_check = rev / cnt if cnt else 0
    print(f"{cat:<20} | {cnt:>10} | {rev:>15,.0f} | {avg_check:>12,.0f}")


#4
month_counts = {}
for row in rows:
    month = row["order_date"].strip()[:7]
    month_counts[month] = month_counts.get(month, 0) + 1

for month in sorted(month_counts):
    print(f"{month}: {month_counts[month]} заказов")

Одежда: 1529
Электроника: 1185
Дом: 1105
Красота: 991
Книги: 653
Категория            |   Оплачено |      Выручка, ₸ |   Ср. чек, ₸
-----------------------------------------------------------------
Электроника          |       1065 |      65,270,240 |       61,287
Одежда               |       1379 |      18,321,080 |       13,286
Дом                  |        987 |      10,516,320 |       10,655
Красота              |        890 |       5,507,620 |        6,188
Книги                |        583 |       2,294,100 |        3,935
2026-09: 2782 заказов
2026-10: 2681 заказов


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [9]:
# Стандарт
#5

month_revenue = {}
for row in rows:
    if row["status"].strip().lower() in ("paid", "оплачен"):
        month = row["order_date"].strip()[:7]
        month_revenue[month] = month_revenue.get(month, 0.0) + to_num(row["amount"])

for month in sorted(month_revenue):
    print(f"{month}: {month_revenue[month]:,.0f} ₸")

# Находим ключи для сентября (-09) и октября (-10)
sep_rev = sum(v for k, v in month_revenue.items() if k.endswith("-09"))
oct_rev = sum(v for k, v in month_revenue.items() if k.endswith("-10"))

if sep_rev > 0:
    diff_pct = (oct_rev - sep_rev) / sep_rev * 100
    print(f"\nИзменение октября к сентябрю: {diff_pct:+.1f}%")
print("\n")

#6

city_total = {}
city_cancelled = {}

for row in rows:
    city = clean_city(row["city"])
    status = row["status"].strip().lower()
    
    city_total[city] = city_total.get(city, 0) + 1
    if status in ("cancelled", "canceled", "отменён", "отменен"):
        city_cancelled[city] = city_cancelled.get(city, 0) + 1

for city in sorted(city_total):
    total = city_total[city]
    canc = city_cancelled.get(city, 0)
    share = (canc / total) * 100 if total else 0
    print(f"{city:<12}: {share:.1f}% ({canc} из {total})")
print("\n")

#7

cust_revenue = {}
cust_orders = {}

for row in rows:
    if row["status"].strip().lower() in ("paid", "оплачен"):
        cid = row["customer_id"].strip()
        cust_revenue[cid] = cust_revenue.get(cid, 0.0) + to_num(row["amount"])
        cust_orders[cid] = cust_orders.get(cid, 0) + 1

top5 = sorted(cust_revenue.items(), key=lambda x: x[1], reverse=True)[:5]

for i, (cid, rev) in enumerate(top5, 1):
    print(f"{i}. Клиент {cid}: {rev:,.0f} ₸ (оплаченных заказов: {cust_orders[cid]})")
print("\n")

#8

city_customers = {}

for row in rows:
    city = clean_city(row["city"])
    cid = row["customer_id"].strip()
    if cid == "TEST":  # тестовый клиент — не настоящий покупатель
        continue

    # setdefault создаёт пустое множество set(), если города ещё нет в словаре
    city_customers.setdefault(city, set()).add(cid)

for city, customers in sorted(city_customers.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{city:<12}: {len(customers)} уникальных клиентов")
print("\n")

2026-09: 57,489,850 ₸
2026-10: 44,419,510 ₸

Изменение октября к сентябрю: -22.7%


Актобе      : 10.3% (41 из 397)
Алматы      : 7.9% (172 из 2188)
Астана      : 8.8% (123 из 1390)
Атырау      : 7.5% (11 из 146)
Караганда   : 8.9% (48 из 538)
Шымкент     : 20.4% (164 из 804)


1. Клиент TEST: 29,999,970 ₸ (оплаченных заказов: 3)
2. Клиент C01627: 257,180 ₸ (оплаченных заказов: 7)
3. Клиент C00485: 215,920 ₸ (оплаченных заказов: 6)
4. Клиент C01338: 192,080 ₸ (оплаченных заказов: 6)
5. Клиент C02122: 191,670 ₸ (оплаченных заказов: 5)


Алматы      : 1351 уникальных клиентов
Астана      : 981 уникальных клиентов
Шымкент     : 655 уникальных клиентов
Караганда   : 451 уникальных клиентов
Актобе      : 358 уникальных клиентов
Атырау      : 138 уникальных клиентов




**Ответ к №7:** 

Цифры: у TEST 3 заказа по 9 999 990 ₸, всего около 30 млн ₸. Это больше, чем у всех остальных из топ-5 вместе, и примерно в 100 раз больше обычного клиента.
Почему это важно: два из трёх заказов TEST сделаны в сентябре, один в октябре. Значит, они искажают выручку по месяцам и процент −22.7% из №5. Насколько именно, ты как раз посчитаешь в №10.
Вывод: такие строки нужно убирать из данных перед расчётами.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж